# What Makes Chicago Crashes Severe?

CS 418 Introduction to Data Science, Progress Report

Team Data Driven: Salwa Majeed, Axel Alvarez, Hibatul Muqeet, Yaswin Chebrolu

GitHub repository: https://github.com/YaswinChebrolu/Chicago-Traffic-Crash-Trends

How to use this file: every section below has a short prompt in plain text. Replace the prompt with your own writing, put your name next to anything you built, and keep the section order. Code cells are marked with a comment saying whose they are.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# This notebook lives in the notebooks folder, so the data path goes up one level.
# Download cleaned_crashes.csv.gz from the team Drive link and put it in the data folder.
df = pd.read_csv("../data/cleaned_crashes.csv.gz", parse_dates=["crash_date"], low_memory=False)
print(df.shape)
df.head()

## 1. Project introduction

Write 1 or 2 short paragraphs. Cover the data, the question, why it matters, and the hypotheses.

Data: Chicago Data Portal, Traffic Crashes, Crashes dataset (reported crashes in Chicago).

Main question: What makes a Chicago traffic crash severe, and where and when are the most dangerous crashes?

Definition: a crash is severe if it involved a fatal or incapacitating injury.

Why it matters: [write 2 or 3 sentences about road safety and where improvements could help]

Our hypotheses:

H1: Higher speed limits lead to more severe crashes.

H2: Darkness leads to more severe crashes.

H3: Bad weather raises crash counts but not severity.

H4: Hit and run crashes are more common at night.

## 2. Changes since the proposal

Parts removed from our plan:

[for example, anything we dropped, such as the People or Vehicles tables if we decided not to use them]

Parts added to our plan:

[for example, keeping only 2018 onward, defining severe as fatal or incapacitating, adding NOAA weather data, adding a speed limit analysis]

Why the scope changed:

[1 or 2 sentences]

## 3. Data

Data section (Salwa Majeed)

Source: Chicago Data Portal, Traffic Crashes, Crashes dataset.

What we did: We downloaded 1,108,694 crash records. We removed duplicates and rows with no injury information, leaving 1,106,297. We kept crashes from 2018 onward because earlier years (2013 to 2017) have far fewer records, which suggests incomplete reporting. That left 968,596 crashes from 2018 through October 2026 (2026 is a partial year).

Cleaning: We turned UNKNOWN values into blanks, set impossible coordinates (0) to blank, and set speed limits outside 5 to 70 mph to blank. After cleaning, about 5.2 percent of rows are missing lighting, 6.5 percent are missing weather, 0.5 percent are missing speed limit, and about 0.8 percent are missing coordinates.

New columns: severe (1 if the crash had a fatal or incapacitating injury, 0 otherwise), any_injury, hit_and_run, and crash_year. About 1.79 percent of crashes are severe.

Limits: The data has no traffic volume, so we compare severity rates and not crash counts. Freeway crashes are largely missing, and only reported crashes are included. The newest crashes may be missing later fatality updates.

Cleaning code: see notebooks/salwa_data_and_stats.ipynb in the repository.

Other data we added (name the owner): [for example, NOAA daily weather for Chicago, added by Member 3]

In [ ]:
# Data summary (Salwa)
print(df.shape)
print(df["crash_year"].value_counts().sort_index())
print(df["severe"].mean())

## 4. Exploratory data analysis

Show what the data looks like and note anything interesting or odd. Suggested pieces: crashes per year, crashes by hour of day and day of week, the share of severe crashes, and anything surprising.

Stats tests (Salwa Majeed): We ran chi square tests of severity against lighting, speed limit, and weather. All three were statistically significant, but the links were weak (Cramer's V of 0.035 for lighting, 0.034 for speed limit, and 0.011 for weather). Lighting and speed limit had the strongest links, and weather had the weakest. Each factor on its own explains only a small part of severity.

Weather results (Salwa Majeed): Severe crash rates were similar across most weather conditions (about 1.6 to 2.2 percent). Rain was slightly higher than clear weather (2.16 versus 1.89 percent), and snow was lower (1.24 percent). This is consistent with the severity part of H3, but H3 also says bad weather raises the number of crashes, which will be tested separately in the weather and forecasting analysis. Categories with fewer than about 1,000 crashes were not used for conclusions because their rates are unreliable.

In [ ]:
# Chi square tests (Salwa)
def chi_test(col):
    table = pd.crosstab(df[col], df["severe"])
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.to_numpy().sum()
    cramers_v = (chi2 / (n * (min(table.shape) - 1))) ** 0.5
    return chi2, p, cramers_v

for col in ["lighting_condition", "posted_speed_limit", "weather_condition"]:
    chi2, p, v = chi_test(col)
    print(col, "| chi2 =", round(chi2, 1), "| p =", p, "| Cramer's V =", round(v, 3))

In [ ]:
# Other exploration charts go here (add the owner's name in a comment)


## 5. Visualizations

At least five. For each one: the hypothesis it tests, why we thought it was interesting, the member responsible, the chart, and what it shows.

Viz 1 (Salwa Majeed): Crashes in daylight were the least likely to be severe (1.57 percent). Crashes at dawn, dusk, and in darkness were all more likely to be severe. Darkness on a lighted road had the highest rate (2.70 percent), higher than darkness on an unlit road (1.97 percent). This partly supports H2. The lighted road result may mean road type or traffic speed matters too, which we can look at with our models.

In [ ]:
# Viz 1 (Salwa)
stats = df.groupby("lighting_condition")["severe"].agg(["mean", "count"])
stats = stats[stats["count"] >= 1000]
rate = (stats["mean"] * 100).sort_values()

ax = rate.plot(kind="barh", figsize=(8, 5))
ax.bar_label(ax.containers[0], fmt="%.2f%%")
ax.set_xlabel("Percent of crashes that were severe")
ax.set_ylabel("Lighting condition")
ax.set_title("Severe crash rate by lighting condition")
plt.tight_layout()
plt.show()

Viz 1b (Salwa Majeed): The severe crash rate generally rises with the posted speed limit, from about 0.6 percent at 10 mph to 1.86 percent at 30 mph and 3.58 percent at 40 mph. This supports H1, although the increase is not perfectly smooth above 40 mph. Speed limits with fewer than 300 crashes were left out because their rates are unreliable.

In [ ]:
# Viz 1b (Salwa)
speed = df.groupby("posted_speed_limit")["severe"].agg(["mean", "count"])
speed = speed[speed["count"] >= 300]
speed_rate = speed["mean"] * 100
speed_rate.index = speed_rate.index.astype(int)

ax = speed_rate.plot(kind="bar", figsize=(9, 5))
ax.set_xlabel("Posted speed limit (mph)")
ax.set_ylabel("Percent of crashes that were severe")
ax.set_title("Severe crash rate by posted speed limit")
ax.bar_label(ax.containers[0], fmt="%.2f")
plt.tight_layout()
plt.show()

Viz 2 (member name): [Hypothesis tested, why it is interesting, and what the chart shows. Suggested: which features matter most for predicting severe crashes.]

In [ ]:
# Viz 2 (member name)


Viz 3 (member name): [Hypothesis tested, why it is interesting, and what the chart shows. Suggested: crash counts and severe share by weather condition, for H3.]

In [ ]:
# Viz 3 (member name)


Viz 4 (member name): [Hypothesis tested, why it is interesting, and what the chart shows. Suggested: a map of crash hotspots.]

In [ ]:
# Viz 4 (member name)


Viz 5 (member name): [Hypothesis tested, why it is interesting, and what the chart shows. Suggested: hit and run share by hour of day, for H4.]

In [ ]:
# Viz 5 (member name)


## 6. Machine learning analyses

At least two. For each one: the member responsible, the question, the baseline, the model, the results, and a plain language interpretation.

### ML analysis 1 (member name)

Question: [for example, can we predict whether a crash is severe from conditions known at the time of the crash?]

Features used: [list them, and note that injury, damage, and contributory cause columns were left out because they are only known after the crash]

Baseline: [for example, always predict not severe]

Models: [for example, logistic regression and random forest]

Results: [a small table of precision, recall, F1, and AUC for the baseline and each model]

Interpretation: [what the results mean in plain words, and what the model got right or wrong]

In [ ]:
# ML analysis 1 (member name)


### ML analysis 2 (member name)

Question: [for example, can we forecast daily crash counts using weather and calendar features?]

Features used: [list them]

Baseline: [for example, the average of the previous 7 days]

Model: [name it]

Results: [for example, mean absolute error for the baseline and the model]

Interpretation: [what the results mean in plain words]

In [ ]:
# ML analysis 2 (member name)


## 7. Reflection

What is the most challenging part of the project so far?

[answer]

What are our initial insights?

[answer, for example: speed limit and lighting have the strongest link to severe crashes, weather the weakest, and all links are weak on their own]

Are there concrete results we can show at this point? If not, why not?

[answer]

Going forward, what are the biggest problems we face?

[answer, for example: class imbalance since only about 1.8 percent of crashes are severe, and no traffic volume data]

Are we on track? If not, which parts need more time?

[answer]

Given our initial exploration, is it worth proceeding? If not, how will we move forward?

[answer]

## 8. Next steps

Concrete plans and goals for the next month. Include who owns each item and a target date.

[for example: finish and tune the severity model, finish the forecast and hotspot analyses, build the interactive map dashboard, run any extra tests, and start the final report]